# Lab 2 — Basic Anomaly Detection for Cybersecurity Logs

**Student:** Sharon Weisman

## Lab goal
This notebook builds a small end-to-end anomaly detection pipeline for a cybersecurity-related dataset.  
I used a **synthetic login-events dataset** designed to represent **MITRE ATT&CK T1110 — Brute Force**.

### Why this dataset fits the assignment
The lab requires:
- at least one **time-based feature**
- at least one **numeric feature**
- at least one **categorical feature**
- anomalies as a **small minority**

This dataset includes:
- **Time-based:** `timestamp`, `hour`
- **Numeric:** `failed_logins`, `session_duration_sec`, `bytes_sent`
- **Categorical:** `user`, `city`, `src_ip`
- **Label column for evaluation only:** `label`

The anomalies were intentionally downsampled to about **2%** of the data, which matches the assignment requirement for unsupervised anomaly detection.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("synthetic_login_anomaly_dataset.csv", parse_dates=["timestamp"])
df.head()


## 1. Dataset preparation / loading

This dataset simulates authentication activity in an enterprise environment.

### Normal behavior
Normal events mostly occur during business hours, with a low number of failed login attempts, medium session duration, and moderate bytes sent.

### Anomalous behavior
The anomalous events represent likely brute-force or suspicious login attempts:
- many failed logins
- unusual login hours
- very short sessions
- lower-than-normal bytes sent
- some events from unusual locations or unknown cities


In [ ]:
print("Rows, columns:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nClass ratio:")
print(df["label"].value_counts(normalize=True).round(4))


## 2. Exploratory Data Analysis (EDA)

The assignment requires:
1. basic dataset statistics  
2. at least two visualizations  
3. a short analytical summary of normal patterns and expected anomalies


In [ ]:
df.describe(include="all")


In [ ]:
print("Missing values by column:")
df.isnull().sum()


In [ ]:
# Histogram 1: failed logins
plt.figure()
df["failed_logins"].hist(bins=25)
plt.xlabel("failed_logins")
plt.ylabel("count")
plt.title("Distribution of Failed Logins")
plt.show()

# Histogram 2: session duration
plt.figure()
df["session_duration_sec"].hist(bins=25)
plt.xlabel("session_duration_sec")
plt.ylabel("count")
plt.title("Distribution of Session Duration")
plt.show()


In [ ]:
# Categorical visualization: city counts
city_counts = df["city"].value_counts()
plt.figure()
plt.bar(city_counts.index.astype(str), city_counts.values)
plt.xticks(rotation=45, ha="right")
plt.xlabel("city")
plt.ylabel("count")
plt.title("City Distribution")
plt.tight_layout()
plt.show()


In [ ]:
# Time-based visualization: events by hour
hour_counts = df["hour"].value_counts().sort_index()
plt.figure()
plt.bar(hour_counts.index.astype(int), hour_counts.values)
plt.xlabel("hour")
plt.ylabel("number of events")
plt.title("Events by Hour")
plt.show()


In [ ]:
# Scatter plot of two key numeric features
label_map = {"normal": 0, "attack": 1}
colors = df["label"].map(label_map)

plt.figure()
plt.scatter(df["failed_logins"], df["session_duration_sec"], c=colors)
plt.xlabel("failed_logins")
plt.ylabel("session_duration_sec")
plt.title("Failed Logins vs Session Duration")
plt.show()


### EDA summary

The dataset shows a clear concentration of normal activity during daytime hours, with relatively few failed login attempts and longer sessions.  
Most records also belong to common cities and produce a moderate amount of outbound bytes.  
The expected anomalies are login events that happen at unusual hours, contain a much larger number of failed attempts, and terminate quickly.  
These patterns are consistent with brute-force behavior, where repeated authentication failures occur before a short or abnormal access attempt.


## 3. Apply an anomaly detection model

I use **Isolation Forest** as the main model, as required by the assignment.

### Preprocessing plan
- Encode categorical features with **OneHotEncoder**
- Scale numeric features with **StandardScaler**

### Features used
The model uses:
- numeric: `hour`, `failed_logins`, `session_duration_sec`, `bytes_sent`
- categorical: `user`, `city`, `src_ip`


In [ ]:
feature_cols = ["hour", "failed_logins", "session_duration_sec", "bytes_sent", "user", "city", "src_ip"]
X = df[feature_cols]
y_true = (df["label"] == "attack").astype(int)

numeric_features = ["hour", "failed_logins", "session_duration_sec", "bytes_sent"]
categorical_features = ["user", "city", "src_ip"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", iso_forest)
])

pipeline.fit(X)

raw_pred = pipeline.named_steps["model"].predict(
    pipeline.named_steps["preprocessor"].transform(X)
)
# IsolationForest returns -1 for anomaly and 1 for normal
y_pred = np.where(raw_pred == -1, 1, 0)

# anomaly score: smaller = more abnormal, so invert sign for readability
scores = -pipeline.named_steps["model"].score_samples(
    pipeline.named_steps["preprocessor"].transform(X)
)

df["anomaly_score"] = scores
df["predicted_anomaly"] = y_pred

print("Detected anomalies:", int(df["predicted_anomaly"].sum()))
print("Total rows:", len(df))


In [ ]:
plt.figure()
plt.hist(df["anomaly_score"], bins=30)
plt.xlabel("anomaly_score")
plt.ylabel("count")
plt.title("Histogram of Isolation Forest Anomaly Scores")
plt.show()


In [ ]:
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["normal", "attack"], zero_division=0))


### Interpretation of model results

The model identifies a small minority of records as anomalous, which matches the intended structure of the dataset.  
High anomaly scores are mostly associated with records that have many failed login attempts, unusual hours, and very short sessions.  
Because the dataset was constructed around brute-force-like behavior, the anomalies detected by Isolation Forest are aligned with the expected threat pattern.


## 4. Visualize anomalies on a 2D projection

The assignment asks for a 2D visualization using PCA, t-SNE, or UMAP.  
I use **PCA** as the baseline method.


In [ ]:
X_processed = pipeline.named_steps["preprocessor"].transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed)

plot_df = pd.DataFrame({
    "pc1": X_pca[:, 0],
    "pc2": X_pca[:, 1],
    "predicted_anomaly": df["predicted_anomaly"],
    "label": df["label"],
    "anomaly_score": df["anomaly_score"],
})
plot_df.head()


In [ ]:
# 2D scatter plot colored by predicted anomaly label
plt.figure()
normal_mask = plot_df["predicted_anomaly"] == 0
anomaly_mask = plot_df["predicted_anomaly"] == 1

plt.scatter(plot_df.loc[normal_mask, "pc1"], plot_df.loc[normal_mask, "pc2"], label="Predicted normal")
plt.scatter(plot_df.loc[anomaly_mask, "pc1"], plot_df.loc[anomaly_mask, "pc2"], label="Predicted anomaly")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA Projection of Login Events")
plt.legend()
plt.show()


In [ ]:
# Optional second view: colored by anomaly score
plt.figure()
plt.scatter(plot_df["pc1"], plot_df["pc2"], c=plot_df["anomaly_score"])
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA Projection Colored by Anomaly Score")
plt.show()


### 2D projection explanation

Most normal behavior forms a denser region in the PCA space, while the anomalous records appear farther away from the main concentration of points.  
This supports the intuition that suspicious login events differ from the common behavior across multiple features at the same time.  
In practice, the 2D view helps confirm that the model is not flagging anomalies randomly, but is isolating records that sit on the edges of the data distribution.


## 5. Conclusion

This lab demonstrated a full anomaly detection workflow on cybersecurity-related login data.  
I created a synthetic dataset mapped to **MITRE ATT&CK T1110 (Brute Force)**, performed exploratory data analysis, trained an **Isolation Forest** model, and visualized the results in 2D using **PCA**.  
The model successfully highlighted records with unusual login timing, high failed-login counts, and short sessions, which are reasonable indicators of suspicious authentication behavior.  
Overall, the pipeline shows how unsupervised anomaly detection can support early identification of potentially malicious activity even when labeled data is limited.
